# CycleGAN + CBAM — Colab training notebook

Trains the OCT image-enhancement model on a Colab GPU.

## Before running

1. **Upload your dataset zip to Drive.** From your local project root:
   ```bash
   cd /path/to/medical-gan-oct
   zip -r processed.zip data/processed
   ```
   Then drag `processed.zip` into your Drive at `My Drive/processed.zip`.

2. **Enable GPU.** Runtime → Change runtime type → Hardware accelerator: **GPU**. Save.

## How storage is laid out

- **Data** is unzipped onto Colab's fast local disk (~10× faster reads than Drive).
- **Checkpoints and sample PNGs** are symlinked to Drive so they survive session shutdown.

Run cells top to bottom.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone the repo and install

In [ ]:
%cd /content
![ -d medical-gan-oct ] || git clone https://github.com/BLHmarwane/medical-gan-oct.git
%cd medical-gan-oct
!git pull
!pip install -q -r requirements.txt
!pip install -q -e .

## 4. Unzip the dataset to local Colab disk

Reads from Drive are slow (network mount). Unzipping to `/content/medical-gan-oct/data/processed/`
puts the dataset on Colab's fast local SSD. The zip's internal structure is
`data/processed/domain_A/...`, so we unzip at the repo root.

In [ ]:
import shutil
from pathlib import Path

REPO = Path('/content/medical-gan-oct')
ZIP = '/content/drive/MyDrive/processed.zip'  # adjust if you placed it elsewhere

# Wipe any prior data folder so re-runs are clean
data_dir = REPO / 'data' / 'processed'
if data_dir.is_symlink():
    data_dir.unlink()
elif data_dir.exists():
    shutil.rmtree(data_dir)

assert Path(ZIP).exists(), f'Missing zip at {ZIP}. Upload processed.zip to your Drive first.'
!unzip -q "{ZIP}" -d "{REPO}"
print('Done.')

## 5. Symlink checkpoints and logs to Drive

These get written DURING training; symlinking ensures they persist when the
Colab session ends. (Data does NOT get symlinked — it's read constantly
during training, so it stays on local disk for speed.)

In [ ]:
import shutil
from pathlib import Path

REPO = Path('/content/medical-gan-oct')
DRIVE = Path('/content/drive/MyDrive/medical-gan-oct')

# Make sure Drive folders exist
(DRIVE / 'checkpoints').mkdir(parents=True, exist_ok=True)
(DRIVE / 'logs/samples').mkdir(parents=True, exist_ok=True)

def symlink(src: Path, dst: Path) -> None:
    if dst.is_symlink():
        dst.unlink()
    elif dst.exists():
        shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.symlink_to(src)

symlink(DRIVE / 'checkpoints', REPO / 'checkpoints')
symlink(DRIVE / 'logs',        REPO / 'logs')
print('Symlinks ready.')
!ls -la /content/medical-gan-oct/checkpoints /content/medical-gan-oct/logs

## 6. Sanity-check the dataset

In [ ]:
import os
a = os.listdir('/content/medical-gan-oct/data/processed/domain_A')
b = os.listdir('/content/medical-gan-oct/data/processed/domain_B')
print(f'domain_A: {len(a)} files')
print(f'domain_B: {len(b)} files')
assert len(a) > 0 and len(b) > 0, 'Dataset is empty — re-run cell 4.'

## 7. Train

On T4: ~30s/epoch → ~100 minutes for 200 epochs.
Sample PNGs and checkpoints stream to Drive as they're written.

In [ ]:
!python scripts/train.py --config configs/cyclegan_cbam_colab.yaml

## 8. Preview the latest sample grid

In [ ]:
from pathlib import Path
from IPython.display import Image, display

samples = sorted(Path('/content/medical-gan-oct/logs/samples').glob('epoch_*.png'))
if samples:
    print(f'{len(samples)} sample grids saved. Showing the latest:')
    display(Image(filename=str(samples[-1])))
else:
    print('No samples yet — training may still be starting.')